# LangChain 中间件使用指南

> **说明**: LangChain 没有名为 "Middleware" 的类，但通过 **Callbacks**、**Runnables**、**缓存** 等机制实现了中间件功能

### 本教程涵盖：
1. Callbacks 回调中间件
2. 缓存中间件 (Cache)
3. 重试中间件 (Retry)
4. 降级中间件 (Fallbacks)
5. 流式中间件 (Streaming)
6. 日志中间件 (Logging)
7. 限流中间件 (Rate Limiting)
8. 链式中间件组合

In [1]:
import os
import time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import BaseCallbackHandler
from rich import print as rprint
import dotenv

dotenv.load_dotenv(override=True)

# 基础 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

# 基础链
prompt = ChatPromptTemplate.from_template("用一句话解释：{topic}")
chain = prompt | llm | StrOutputParser()

print("初始化完成")

初始化完成


## 1. Callbacks 回调中间件

回调是最核心的中间件机制，可以拦截 LLM 调用的各个阶段

In [3]:
class TimingCallback(BaseCallbackHandler):
    """计时回调中间件"""
    
    def __init__(self):
        super().__init__()
        self.start_time = None
        self.metrics = []
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.start_time = time.time()
        rprint("[cyan]⏱ 开始计时[/cyan]")
    
    def on_llm_end(self, response, **kwargs):
        if self.start_time:
            elapsed = time.time() - self.start_time
            self.metrics.append(elapsed)
            rprint(f"[green]✓ 完成，耗时: {elapsed:.2f}秒[/green]")
    
    def on_llm_error(self, error, **kwargs):
        rprint(f"[red]✗ 错误: {error}[/red]")

# 使用回调
timing = TimingCallback()
result = chain.invoke({"topic": "Python"}, config={"callbacks": [timing]})
rprint(f"\n结果: {result}")
rprint(f"历史耗时: {timing.metrics}")

⏱ 开始计时

✓ 完成，耗时: 2.54秒

结果: Python 是一种简洁通用、易于上手的编程语言，常被称为“胶水语言”。

历史耗时: [2.544632911682129]

## 2. 缓存中间件 (Cache)

避免重复调用 LLM，节省成本和时间

In [5]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

# 设置内存缓存
set_llm_cache(InMemoryCache())

rprint("[bold]缓存中间件测试[/bold]")

# 第一次调用
start = time.time()
result1 = llm.invoke("什么是机器学习？")
time1 = time.time() - start
rprint(f"第一次调用: {time1:.2f}秒")

# 第二次调用（缓存命中）
start = time.time()
result2 = llm.invoke("什么是机器学习？")
time2 = time.time() - start
rprint(f"第二次调用: {time2:.2f}秒 [green](缓存命中)[/green]")

rprint(f"加速比: {time1/max(time2, 0.001):.1f}倍")

# 清除缓存
set_llm_cache(None)

缓存中间件测试

第一次调用: 9.56秒

第二次调用: 0.00秒 (缓存命中)

加速比: 4755.1倍

## 3. 重试中间件 (Retry)

自动重试失败的请求，支持指数退避

In [7]:
# 重试中间件
llm_with_retry = llm.with_retry(
    stop_after_attempt=3,        # 最多重试3次
    wait_exponential_jitter=True  # 指数退避+抖动
)

chain_with_retry = prompt | llm_with_retry | StrOutputParser()

rprint("[bold]重试中间件测试[/bold]")
result = chain_with_retry.invoke({"topic": "深度学习"})
rprint(f"结果: {result}")

重试中间件测试

结果: 深度学习是通过多层神经网络，让机器像孩子学认猫一样，从原始数据中逐层抽象出特征并做出预测的方法。

## 4. 降级中间件 (Fallbacks)

主模型失败时自动切换到备用模型

In [8]:
# 主模型（模拟失败）
primary_llm = ChatOpenAI(
    model="nonexistent-model",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

# 备用模型
fallback_llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

# 降级链
chain_with_fallback = prompt | primary_llm.with_fallbacks([fallback_llm]) | StrOutputParser()

rprint("[bold]降级中间件测试[/bold]")
try:
    result = chain_with_fallback.invoke({"topic": "AI"})
    rprint(f"[green]降级成功:[/green] {result}")
except Exception as e:
    rprint(f"[red]错误:[/red] {e}")

降级中间件测试

降级成功: **AI（人工智能）是让机器模拟人类智能（如学习、推理、感知）的技术。**

## 5. 流式中间件 (Streaming)

实时输出 token，提升用户体验

In [9]:
class StreamCallback(BaseCallbackHandler):
    """流式输出回调"""
    
    def on_llm_new_token(self, token, **kwargs):
        print(token, end="", flush=True)

rprint("[bold]流式输出:[/bold]")
print("回答: ", end="")

# 使用流式回调
llm_stream = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    streaming=True,
    callbacks=[StreamCallback()]
)

llm_stream.invoke("用一句话介绍Python")
print()

流式输出:

回答: Python是一门简洁易学、功能强大且应用广泛的编程语言。


## 6. 日志中间件 (Logging)

记录所有 LLM 调用的详细信息

In [10]:
from datetime import datetime

class LoggingCallback(BaseCallbackHandler):
    """日志记录中间件"""
    
    def __init__(self):
        super().__init__()
        self.logs = []
    
    def _log(self, event, details):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "event": event,
            "details": details
        }
        self.logs.append(entry)
        rprint(f"[dim][{entry['timestamp']}] {event}: {details}[/dim]")
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        self._log("LLM_START", f"输入: {prompts[0][:50]}...")
    
    def on_llm_end(self, response, **kwargs):
        self._log("LLM_END", f"完成")
    
    def on_llm_error(self, error, **kwargs):
        self._log("LLM_ERROR", str(error))
    
    def on_chain_start(self, serialized, inputs, **kwargs):
        self._log("CHAIN_START", serialized.get('name', 'unknown'))
    
    def on_chain_end(self, outputs, **kwargs):
        self._log("CHAIN_END", "完成")

# 使用日志中间件
logger = LoggingCallback()
result = chain.invoke(
    {"topic": "机器学习"},
    config={"callbacks": [logger]}
)

rprint(f"\n[bold]日志记录 ({len(logger.logs)} 条):[/bold]")
for log in logger.logs[-3:]:
    rprint(f"  {log['event']}: {log['details']}")

Error in LoggingCallback.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


[2026-07-06T21:16:58.191810] CHAIN_START: ChatPromptTemplate

[2026-07-06T21:16:58.193853] CHAIN_END: 完成

[2026-07-06T21:16:58.197810] LLM_START: 输入: Human: 用一句话解释：机器学习...

[2026-07-06T21:17:00.438094] LLM_END: 完成

...

[2026-07-06T21:17:00.447117] CHAIN_END: 完成

日志记录 (6 条):

LLM_END: 完成

CHAIN_END: 完成

CHAIN_END: 完成

## 7. 限流中间件 (Rate Limiting)

控制请求频率，避免超出 API 限制

In [11]:
import threading

class RateLimitCallback(BaseCallbackHandler):
    """限流中间件"""
    
    def __init__(self, max_calls_per_minute=10):
        super().__init__()
        self.max_calls = max_calls_per_minute
        self.calls = []
        self.lock = threading.Lock()
    
    def _check_rate(self):
        with self.lock:
            now = time.time()
            # 清理超过1分钟的记录
            self.calls = [t for t in self.calls if now - t < 60]
            
            if len(self.calls) >= self.max_calls:
                wait_time = 60 - (now - self.calls[0])
                rprint(f"[yellow]⚠ 限流: 等待 {wait_time:.1f}秒[/yellow]")
                time.sleep(wait_time)
            
            self.calls.append(time.time())
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        self._check_rate()

# 使用限流中间件（每分钟最多2次调用）
rate_limiter = RateLimitCallback(max_calls_per_minute=2)

rprint("[bold]限流中间件测试 (每分钟2次)[/bold]")
for i in range(3):
    rprint(f"\n调用 {i+1}:")
    result = llm.invoke("你好", config={"callbacks": [rate_limiter]})
    rprint(f"  结果: {result.content[:30]}...")

限流中间件测试 (每分钟2次)

调用 1:

结果: 你好！很高兴见到你！😊 我是MiMo-v2.5，由小米大模型...

调用 2:

结果: 你好！我是 MiMo，由小米大模型 Core 团队开发的智能...

调用 3:

⚠ 限流: 等待 53.2秒

结果: 你好！😊 我是 **MiMo-v2.5**，由小米大模型Co...

## 8. 链式中间件组合

将多个中间件组合使用，构建完整的处理管道

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableConfig

# 1. 输入验证中间件
def validate_input(input):
    if not input.get("topic"):
        raise ValueError("缺少 topic 参数")
    if len(input["topic"]) > 100:
        raise ValueError("topic 过长")
    return input

# 2. 输出格式化中间件
def format_output(output):
    return {
        "answer": output,
        "length": len(output),
        "timestamp": datetime.now().isoformat()
    }

# 3. 组合中间件链
full_chain = (
    RunnableLambda(validate_input)   # 输入验证
    | prompt                          # 模板
    | llm.with_retry(stop_after_attempt=2)  # 重试
    | StrOutputParser()               # 解析
    | RunnableLambda(format_output)   # 输出格式化
)

# 4. 使用组合中间件
callbacks = [TimingCallback(), LoggingCallback()]
config = RunnableConfig(callbacks=callbacks)

rprint("[bold]链式中间件组合[/bold]")
result = full_chain.invoke({"topic": "神经网络"}, config=config)

rprint(f"\n[green]最终结果:[/green]")
rprint(f"  回答: {result['answer'][:60]}...")
rprint(f"  长度: {result['length']} 字符")
rprint(f"  时间: {result['timestamp']}")

## 9. 自定义中间件基类

创建可复用的中间件基类

In [ ]:
from abc import ABC, abstractmethod

class BaseMiddleware(BaseCallbackHandler, ABC):
    """中间件基类"""
    
    def __init__(self, name="BaseMiddleware"):
        super().__init__()
        self.name = name
        self.enabled = True
    
    def enable(self):
        self.enabled = True
        rprint(f"[green]{self.name} 已启用[/green]")
    
    def disable(self):
        self.enabled = False
        rprint(f"[yellow]{self.name} 已禁用[/yellow]")

class MetricsMiddleware(BaseMiddleware):
    """指标收集中间件"""
    
    def __init__(self):
        super().__init__("MetricsMiddleware")
        self.total_calls = 0
        self.total_tokens = 0
        self.errors = 0
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        if self.enabled:
            self.total_calls += 1
    
    def on_llm_end(self, response, **kwargs):
        if self.enabled and response.llm_output:
            usage = response.llm_output.get('token_usage', {})
            self.total_tokens += usage.get('total_tokens', 0)
    
    def on_llm_error(self, error, **kwargs):
        if self.enabled:
            self.errors += 1
    
    def get_metrics(self):
        return {
            "total_calls": self.total_calls,
            "total_tokens": self.total_tokens,
            "errors": self.errors,
            "error_rate": f"{self.errors/max(self.total_calls, 1)*100:.1f}%"
        }

# 使用自定义中间件
metrics = MetricsMiddleware()

rprint("[bold]自定义中间件测试[/bold]")
for topic in ["Python", "AI", "机器学习"]:
    chain.invoke({"topic": topic}, config={"callbacks": [metrics]})

rprint(f"\n[green]指标统计:[/green]")
for k, v in metrics.get_metrics().items():
    rprint(f"  {k}: {v}")

## 10. 中间件组合器

将多个中间件组合成一个

In [ ]:
class MiddlewareChain(BaseCallbackHandler):
    """中间件组合器"""
    
    def __init__(self, middlewares=None):
        super().__init__()
        self.middlewares = middlewares or []
    
    def add(self, middleware):
        self.middlewares.append(middleware)
        return self
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        for m in self.middlewares:
            m.on_llm_start(serialized, prompts, **kwargs)
    
    def on_llm_end(self, response, **kwargs):
        for m in self.middlewares:
            m.on_llm_end(response, **kwargs)
    
    def on_llm_error(self, error, **kwargs):
        for m in self.middlewares:
            m.on_llm_error(error, **kwargs)

# 组合多个中间件
middleware_stack = MiddlewareChain()
middleware_stack.add(TimingCallback())
middleware_stack.add(LoggingCallback())
middleware_stack.add(MetricsMiddleware())

rprint("[bold]中间件组合器测试[/bold]")
result = chain.invoke({"topic": "深度学习"}, config={"callbacks": [middleware_stack]})
rprint(f"\n结果: {result[:80]}...")

## 总结

### LangChain 中间件机制

| 机制 | 实现方式 | 用途 |
|------|---------|------|
| 回调 | `BaseCallbackHandler` | 拦截事件 |
| 缓存 | `InMemoryCache` | 避免重复调用 |
| 重试 | `with_retry()` | 自动重试 |
| 降级 | `with_fallbacks()` | 失败切换 |
| 流式 | `streaming=True` | 实时输出 |
| 限流 | 自定义回调 | 控制频率 |
| 日志 | 自定义回调 | 记录调用 |
| 监控 | 自定义回调 | 收集指标 |

### 核心要点
1. **Callbacks** 是 LangChain 中间件的核心
2. **Runnables** 提供链式组合能力
3. 可以自由组合多个中间件
4. 支持同步和异步操作

## 11. 自定义中间件 - 类实现

使用类继承 `BaseCallbackHandler` 实现中间件

In [12]:
from langchain_core.outputs import LLMResult

# ==================== 类实现中间件 ====================

class CacheMiddleware(BaseCallbackHandler):
    """缓存中间件 - 类实现"""
    
    def __init__(self):
        super().__init__()
        self.cache = {}
        self.hits = 0
        self.misses = 0
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        prompt_key = prompts[0] if prompts else ""
        if prompt_key in self.cache:
            self.hits += 1
            rprint(f"[green]✓ 缓存命中[/green]")
        else:
            self.misses += 1
            rprint(f"[yellow]○ 缓存未命中[/yellow]")
    
    def on_llm_end(self, response: LLMResult, **kwargs):
        if response.llm_output:
            pass  # 实际应用中可存储结果
    
    def get_stats(self):
        total = self.hits + self.misses
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": f"{self.hits/max(total,1)*100:.1f}%"
        }

class InputSanitizer(BaseCallbackHandler):
    """输入清理中间件 - 类实现"""
    
    def __init__(self, max_length=500):
        super().__init__()
        self.max_length = max_length
        self.sanitized_count = 0
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        if prompts:
            prompt = prompts[0]
            if len(prompt) > self.max_length:
                self.sanitized_count += 1
                rprint(f"[yellow]⚠ 输入过长，已截断 ({len(prompt)} -> {self.max_length})[/yellow]")

class OutputFilter(BaseCallbackHandler):
    """输出过滤中间件 - 类实现"""
    
    def __init__(self, filter_words=None):
        super().__init__()
        self.filter_words = filter_words or []
        self.filtered_count = 0
    
    def on_llm_end(self, response: LLMResult, **kwargs):
        if response.generations:
            text = response.generations[0][0].text
            for word in self.filter_words:
                if word in text:
                    self.filtered_count += 1
                    rprint(f"[yellow]⚠ 发现敏感词: {word}[/yellow]")

# 测试类实现中间件
rprint("[bold]类实现中间件测试[/bold]\n")

cache_mw = CacheMiddleware()
sanitizer = InputSanitizer(max_length=100)
filter_mw = OutputFilter(filter_words=["错误", "失败"])

# 组合使用
config = {"callbacks": [cache_mw, sanitizer, filter_mw]}
result = chain.invoke({"topic": "Python编程"}, config=config)

rprint(f"\n[green]缓存统计:[/green] {cache_mw.get_stats()}")
rprint(f"[green]清理次数:[/green] {sanitizer.sanitized_count}")

类实现中间件测试

○ 缓存未命中

缓存统计: {'hits': 0, 'misses': 1, 'hit_rate': '0.0%'}

清理次数: 0

## 12. 自定义中间件 - 装饰器实现

使用装饰器模式实现中间件，更简洁灵活

In [13]:
import functools
from typing import Callable, Any

# ==================== 装饰器实现中间件 ====================

def timing_middleware(func: Callable) -> Callable:
    """计时装饰器中间件"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        rprint("[cyan]⏱ 开始计时[/cyan]")
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        rprint(f"[green]✓ 完成，耗时: {elapsed:.2f}秒[/green]")
        return result
    return wrapper

def logging_middleware(func: Callable) -> Callable:
    """日志装饰器中间件"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        rprint(f"[dim]📝 调用: {func.__name__}[/dim]")
        rprint(f"[dim]   参数: {args[1:] if len(args) > 1 else 'N/A'}[/dim]")
        result = func(*args, **kwargs)
        rprint(f"[dim]   返回: {str(result)[:50]}...[/dim]")
        return result
    return wrapper

def error_handler_middleware(func: Callable) -> Callable:
    """错误处理装饰器中间件"""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            rprint(f"[red]✗ 错误: {e}[/red]")
            return f"处理失败: {str(e)}"
    return wrapper

def retry_middleware(max_attempts: int = 3) -> Callable:
    """重试装饰器中间件（带参数）"""
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt < max_attempts - 1:
                        rprint(f"[yellow]⚠ 重试 {attempt + 1}/{max_attempts}[/yellow]")
                        time.sleep(1)
                    else:
                        raise
        return wrapper
    return decorator

def cache_middleware(ttl_seconds: int = 300) -> Callable:
    """缓存装饰器中间件（带参数）"""
    def decorator(func: Callable) -> Callable:
        cache = {}
        
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = str(args) + str(kwargs)
            now = time.time()
            
            if key in cache:
                result, timestamp = cache[key]
                if now - timestamp < ttl_seconds:
                    rprint("[green]✓ 缓存命中[/green]")
                    return result
            
            rprint("[yellow]○ 缓存未命中[/yellow]")
            result = func(*args, **kwargs)
            cache[key] = (result, now)
            return result
        
        wrapper.cache_clear = lambda: cache.clear()
        return wrapper
    return decorator

# 测试装饰器中间件
rprint("[bold]装饰器中间件测试[/bold]\n")

# 应用装饰器
@timing_middleware
@logging_middleware
@error_handler_middleware
def invoke_chain(topic: str):
    return chain.invoke({"topic": topic})

# 调用
result = invoke_chain("机器学习")
rprint(f"\n[green]结果:[/green] {result[:60]}...")

装饰器中间件测试

⏱ 开始计时

📝 调用: invoke_chain

   参数: N/A

   返回: 机器学习是让计算机程序通过数据学习规律，从而在预测或决策中自我优化的科学。...

✓ 完成，耗时: 2.30秒

结果: 机器学习是让计算机程序通过数据学习规律，从而在预测或决策中自我优化的科学。...

## 13. 带参数的装饰器中间件

使用带参数的装饰器实现更灵活的中间件

In [14]:
# ==================== 带参数的装饰器中间件 ====================

# 使用重试装饰器
@retry_middleware(max_attempts=3)
def invoke_with_retry(topic: str):
    return chain.invoke({"topic": topic})

# 使用缓存装饰器
@cache_middleware(ttl_seconds=60)
def invoke_with_cache(topic: str):
    return chain.invoke({"topic": topic})

rprint("[bold]带参数装饰器测试[/bold]\n")

# 测试重试装饰器
rprint("[cyan]测试重试装饰器:[/cyan]")
result = invoke_with_retry("深度学习")
rprint(f"结果: {result[:50]}...\n")

# 测试缓存装饰器
rprint("[cyan]测试缓存装饰器:[/cyan]")
result1 = invoke_with_cache("神经网络")
rprint(f"第一次调用: {result1[:30]}...")

result2 = invoke_with_cache("神经网络")
rprint(f"第二次调用: {result2[:30]}... (应命中缓存)")

带参数装饰器测试

测试重试装饰器:

结果: 深度学习是一种通过多层神经网络从数据中自动学习特征和规律的机器学习方法。...

测试缓存装饰器:

○ 缓存未命中

第一次调用: 神经网络是一种模仿大脑结构的数学模型，通过大量数据学习并做出...

✓ 缓存命中

第二次调用: 神经网络是一种模仿大脑结构的数学模型，通过大量数据学习并做出... (应命中缓存)

## 14. 类 vs 装饰器对比

In [15]:
# ==================== 类 vs 装饰器对比 ====================

rprint("[bold]类 vs 装饰器中间件对比[/bold]\n")

# 类实现 - 更适合复杂逻辑
class ClassMiddleware(BaseCallbackHandler):
    def __init__(self):
        super().__init__()
        self.state = {}  # 可维护状态
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.state['last_prompt'] = prompts[0]
    
    def on_llm_end(self, response, **kwargs):
        self.state['last_response'] = response

# 装饰器实现 - 更简洁
def simple_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

# 混合使用示例
@timing_middleware  # 装饰器
def process_with_both(topic: str):
    # 类中间件通过 config 传入
    class_mw = ClassMiddleware()
    return chain.invoke({"topic": topic}, config={"callbacks": [class_mw]})

rprint("[cyan]混合使用示例:[/cyan]")
result = process_with_both("自然语言处理")
rprint(f"结果: {result[:50]}...")

rprint("""
[bold]选择建议:[/bold]

[cyan]类实现适合:[/cyan]
- 需要维护状态
- 复杂的生命周期管理
- 需要注册多个回调事件
- 需要与其他 LangChain 组件集成

[green]装饰器适合:[/green]
- 简单的横切关注点
- 无需维护状态
- 函数级别的拦截
- 更简洁的代码风格
""")

类 vs 装饰器中间件对比

混合使用示例:

⏱ 开始计时

✓ 完成，耗时: 2.40秒

结果: 自然语言处理是让计算机理解、解释和生成人类语言，以便进行人机交互的技术。...

选择建议:

类实现适合:
- 需要维护状态
- 复杂的生命周期管理
- 需要注册多个回调事件
- 需要与其他 LangChain 组件集成

装饰器适合:
- 简单的横切关注点
- 无需维护状态
- 函数级别的拦截
- 更简洁的代码风格